# MODIS discovery — Guanajuato (earthaccess + pyhdf)

This notebook is a **scientific smoke test**, not an output certified for modeling.

**Location:** 20.4243195, -101.1466639 (regional reference, not a surveyed plot).

The previous FarmAdapt V2 used **ORNL REST JSON** for MODIS; its extraction did not require reading HDF4. Here we retain NASA CMR acquisition with **earthaccess** and read **native LP DAAC HDF4** directly using **pyhdf**, without GDAL's optional HDF4 plugin.

From the repository root install: `python -m pip install -e '.[source,source-test,source-modis]'`. Earthdata login is required only for live downloads.

**No network requests are made unless you set RUN_LIVE=True.**

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from gradientdescentfarmers.source_builder.adapter import (
    REPO_ROOT, load_source_config, NASAProfileSourceBuilder
)
from gradientdescentfarmers.source_builder.core import modis_earthaccess as modis
from gradientdescentfarmers.source_builder.core.preprocessing import (
    standardize_mod13, standardize_mod11, standardize_mod16
)
from gradientdescentfarmers.source_builder.core.validation import validate_table

LAT, LON = 20.4243195, -101.1466639
print("Repo:", REPO_ROOT)
print("Reference:", LAT, LON)

## 1 — Native reader and sinusoidal coordinate test

The complete MODLAND sinusoidal grid uses radius 6371007.181 m. We verify the h/v tile and the containing row/column before accessing an SDS. HDF4 support in rasterio is **not** required.

In [ ]:
import pyhdf
import pyproj
print("pyhdf:", pyhdf.__file__)
print("pyproj:", pyproj.__version__)
x, y = modis._sinusoidal_xy(LAT, LON)
h = int(np.floor((x - modis.X_MIN_M)/modis.TILE_M))
v = int(np.floor((modis.Y_MAX_M - y)/modis.TILE_M))
tile = f"h{h:02d}v{v:02d}"
test_file = Path(f"MOD13Q1.A2024185.{tile}.061.TEST.hdf")
print("Sinusoidal xy (m):", x, y, "Tile:", tile)
for n in (1200, 2400, 4800):
    print("SDS", n, modis.pixel_for_point(test_file, LAT, LON, (n, n)))

## 2 — Inspect existing cached MODIS HDF4 granules (no download)

Looks only under the repository's ignored `data/cache/earthdata/modis/` folder. If nothing is there, proceed to section 3 and explicitly enable the pilot.

In [ ]:
cache = REPO_ROOT / "data/cache/earthdata/modis"
products = ("MOD13Q1", "MOD11A2", "MOD16A2GF")
cached = {product: sorted((cache/product).glob("**/*.hdf")) for product in products}
for product, paths in cached.items():
    print(product, len(paths), "files")
    for path in paths[:2]:
        print(" ", path.relative_to(REPO_ROOT), f"{path.stat().st_size:,} bytes")

## 3 — Live pilot (requires explicit opt-in)

**Start with one MOD13Q1 granule**, not the entire 2024–2025 series. This may download a large regional HDF4 tile. It does **not** overwrite the older `data/processed/` or GPM/SMAP runs.

Set `RUN_LIVE = True` only when ready. Then inspect `availability.json` if a source fails. For all 3 products set `RUN_ALL_PRODUCTS = True` after MOD13Q1 succeeds.

In [ ]:
RUN_LIVE = False
RUN_ALL_PRODUCTS = False
new_run = None

if RUN_LIVE:
    config = load_source_config(REPO_ROOT/"resources/guanajuato_modis_earthaccess.json")
    settings = json.loads((REPO_ROOT/"resources/guanajuato_modis_earthaccess_settings.json").read_text())
    settings["modis_max_granules"] = 1
    for source in ("modis_mod13q1", "modis_mod11a2", "modis_mod16a2gf"):
        settings["source_overrides"][source] = (
            RUN_ALL_PRODUCTS or source == "modis_mod13q1"
        )
    config.pop("settings_file", None)
    config["settings"] = settings
    config["output_root"] = "data/modis_discovery"
    config["run"] = True
    new_run = NASAProfileSourceBuilder().build(config)
    print("Run:", new_run["run_id"])
    print("Status:", new_run["source_status"])
    print("Output:", new_run["output_dir"])
else:
    print("No downloads attempted. Set RUN_LIVE=True to run the pilot.")

## 4 — Inspect SDS names, shapes, raw pixel, and quality flags

Uses **pyhdf** only. A native MODIS file has scientific datasets (SDS), including quality layers. Missing SDS or mismatched tile geometry is reported as an error; no values are invented.

Do not judge science quality by download success alone. QC may reject cloudy or unreliable pixels.

In [ ]:
cached = {product: sorted((cache/product).glob("**/*.hdf")) for product in products}
raw_points = {}
for product, paths in cached.items():
    if not paths:
        print(product, ": no HDF4 yet")
        continue
    path = paths[0]
    print("\nPRODUCT", product, "FILE", path.name)
    datasets = modis.list_science_datasets(path)
    print("SDS:", list(datasets)[:25])
    required = modis.LAYERS[product]
    missing = sorted(set(required.values()) - set(datasets))
    assert not missing, f"Missing native SDS for {product}: {missing}"
    result = modis.extract_point(path, LAT, LON, product)
    raw_points[product] = result
    print(pd.Series({k:v for k,v in result.items() if not k.endswith(("_row","_col","_shape"))}))

## 5 — Apply *original* FarmAdapt transformations and QA checks

The same functions are used for both ORNL and HDF4 pathways; the HDF4 reader supplies raw integers and leaves QA bits unmodified.

In [ ]:
transformers = {
    "MOD13Q1": ("vegetation_timeseries", standardize_mod13),
    "MOD11A2": ("land_surface_temperature", standardize_mod11),
    "MOD16A2GF": ("evapotranspiration", standardize_mod16),
}
for product, record in raw_points.items():
    name, transformer = transformers[product]
    frame = pd.DataFrame([{**record, "farm_id": "guanajuato_valle_demo"}])
    standardized = transformer(frame)
    report = validate_table(name, standardized).as_dict()
    print(product, "| QA passed:", report["passed"],
          "| usable rows:", report["usable_rows"], "| coverage:", report["coverage"])
    print(standardized.head(1).to_string(index=False)[:1500])
    if not report["passed"]:
        print("QA checks:", [c for c in report["checks"] if not c["passed"]])

## 6 — Inspect source errors without another network request

Review the last pilot's availability metadata and the previously failed GPM/SMAP smoke run. The top-level `build_state=failed` is not enough to diagnose authentication, download, SDS, or extraction errors.

In [ ]:
candidate_runs = []
if new_run:
    candidate_runs.append(Path(new_run["output_dir"]))
candidate_runs.append(
    REPO_ROOT/"data/guanajuato_smoke/guanajuato_valle_demo/runs/5508cf72d72d40f4"
)
for run in candidate_runs:
    path = run/"availability.json"
    if not path.is_file():
        print("Not found:", path)
        continue
    statuses = json.loads(path.read_text())
    print("\nRUN:", run.name)
    for source in ("modis_mod13q1", "modis_mod11a2", "modis_mod16a2gf",
                   "gpm_imerg_v07", "smap_l4"):
        rec = statuses.get(source, {})
        if rec.get("status") in ("error", "partial"):
            print(source, rec.get("status"), rec.get("error"), rec.get("warnings"))

## 7 — Decision gate

Proceed to the 2-granule 3-product run only if: download works, pyhdf lists expected SDS, the h/v tile agrees with Guanajuato, native values are extracted and the QA evaluation is auditable.

The notebook itself is exploratory. **One or two granules are not full MODIS temporal coverage.**